# Logarithmic Transformations & Quadratic Terms: Solutions
### Applied Statistical Data Analysis. Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> Complete worked solutions with short interpretations. Compare with your own attempts; the reasoning matters as much as the numbers.

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1: Name the Form (Solution)

| # | Form | Interpretation |
|---|------|----------------|
| a | lin-lin | one unit (percentage point) more market return goes with 1.15 units more fund return: the beta, a unit effect |
| b | log-lin | one percentage point more yield lowers the price by about 9.6% (semi-elasticity; the duration) |
| c | log-log | a 1% larger market cap goes with a 1.08% larger volume: an elasticity, scale-free |
| d | lin-log | a 1% larger AUM goes with 0.45/100 = 0.0045 kCHF, i.e. about 4.5 francs more salary |
| e | log-lin with a dummy | naive: +28%; exact: 100·(e^0.28 − 1) = +32.3% higher wage for CFA holders (Exercise 4 practises this) |

---
# Data for Exercises 2 to 6

In [ ]:
# Provided: build the Swiss cross-section (same code as the Lecture Notebook)
SMI = ['NESN.SW','ROG.SW','NOVN.SW','UBSG.SW','ZURN.SW','CFR.SW','ABBN.SW','SIKA.SW',
       'LONN.SW','ALC.SW','GIVN.SW','HOLN.SW','SLHN.SW','PGHN.SW','SCMN.SW','SREN.SW',
       'GEBN.SW','SOON.SW','LOGN.SW','KNIN.SW']
SMIM = ['BAER.SW','ADEN.SW','CLN.SW','TEMN.SW','VACN.SW','STMN.SW','SCHP.SW','GALE.SW',
        'HELN.SW','PSPN.SW','ALLN.SW','BARN.SW','EMSN.SW','SGSN.SW','LISP.SW','DKSH.SW',
        'SFSN.SW','BUCN.SW','SUN.SW','AVOL.SW']

rows, dropped = [], []
for tick in SMI + SMIM:
    try:
        t  = yf.Ticker(tick)
        px = t.history(period='6mo')
        if len(px) < 60:
            dropped.append((tick, 'short history')); continue
        vol_chf = float((px['Close'] * px['Volume']).mean())
        mcap    = t.fast_info['marketCap']
        if not mcap or mcap <= 0 or vol_chf <= 0:
            dropped.append((tick, 'no market cap / no volume')); continue
        rows.append({'ticker': tick, 'mcap': mcap, 'volume': vol_chf,
                     'D_SMI': 1.0 if tick in SMI else 0.0})
    except Exception as e:              # never swallow silently: n drives every result below
        dropped.append((tick, type(e).__name__))

if dropped:
    print(f'{len(dropped)} tickers dropped: {dropped}')
if not rows:
    raise RuntimeError('No tickers downloaded. Yahoo throttles ~40 sequential Ticker '
                       'calls - wait a minute and re-run, or shorten the list.')

df = pd.DataFrame(rows).set_index('ticker')
df['ln_vol']  = np.log(df['volume'])
df['ln_mcap'] = np.log(df['mcap'])
print(f'n = {len(df)} stocks ({int(df.D_SMI.sum())} SMI)')

---
# Exercise 2: The Elasticity

Run the provided loader cell, then estimate the log-log model WITHOUT the dummy:

$$\ln V_i = \beta_0 + \beta_1 \ln M_i + u_i$$

Report the elasticity with a 95% confidence interval and interpret it in one sentence.

**Written question:** your neighbour estimated the same model on US stocks in dollars. Why are your two elasticities directly comparable even though the currencies differ?

In [ ]:
X1 = sm.add_constant(df['ln_mcap'])
m1 = sm.OLS(df['ln_vol'], X1).fit(cov_type='HC1')
ci = m1.conf_int().loc['ln_mcap']
print(f'elasticity = {m1.params["ln_mcap"]:.3f},  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]')

**Interpretation:** a 1% larger market cap goes with about β₁% larger trading volume. **Comparability:** in the log-log form both sides are in percent, so all units (francs, dollars, billions) cancel. Elasticities are scale-free and therefore comparable across markets and currencies. That is precisely why economists like this form.

---
# Exercise 3: The Dummy, Naive vs Exact

Add the SMI dummy and compute BOTH readings of its coefficient: the naive percentage (100·δ) and the exact percentage effect (100·(e^δ − 1)).

**Written question:** for which magnitudes of δ is the naive reading acceptable, and why exactly does it fail for large δ?

In [ ]:
X2 = sm.add_constant(df[['ln_mcap', 'D_SMI']])
m2 = sm.OLS(df['ln_vol'], X2).fit(cov_type='HC1')
delta = m2.params['D_SMI']
print(f'delta = {delta:.3f} (t = {m2.tvalues["D_SMI"]:.2f})')
print(f'naive reading: {100*delta:+.1f}%')
print(f'exact effect:  {100*(np.exp(delta)-1):+.1f}%')

**Answer:** the naive reading relies on the derivative of the log, which is accurate only for small changes; a dummy jumps from 0 to 1, a large discrete change. For |δ| below about 0.10 the error is negligible (e^0.10 − 1 = 10.5%), beyond that it grows quickly (e^0.35 − 1 = 41.9%, not 35%). Always report the exact effect for economically large dummies.

---
# Exercise 4: Swap the Reference Category

Re-estimate the model with a MID-CAP dummy instead (D_MID = 1 − D_SMI).

(a) What is the new dummy coefficient, and how does it relate to the old one?
(b) Compute the exact percentage effect of being a mid cap. Explain why it is NOT simply minus the SMI effect.
(c) Verify numerically: (1 + g_up)·(1 + g_down) = 1, where g are the two exact effects as decimals.

In [ ]:
df['D_MID'] = 1 - df['D_SMI']
X3 = sm.add_constant(df[['ln_mcap', 'D_MID']])
m3 = sm.OLS(df['ln_vol'], X3).fit(cov_type='HC1')
d_mid = m3.params['D_MID']
d_smi = m2.params['D_SMI']

g_up   = np.exp(d_smi) - 1
g_down = np.exp(d_mid) - 1
print(f'(a) delta_MID = {d_mid:.3f} = -delta_SMI = {-d_smi:.3f}  (same fit, reference swapped)')
print(f'(b) exact mid-cap effect: {100*g_down:+.1f}%   (SMI effect was {100*g_up:+.1f}%)')
print(f'(c) (1 + g_up)(1 + g_down) = {(1+g_up)*(1+g_down):.4f}  → equals 1')

**Answer:** swapping the reference flips the coefficient sign exactly (δ_MID = −δ_SMI), but the exact percentage effects are NOT mirror images: e^δ − 1 and e^−δ − 1 differ in absolute value. They are multiplicative inverses, which is what part (c) verifies. Same asymmetry as returns: +41.9% up and −29.5% down cancel each other.

---
# Exercise 5: RESET as Referee

Estimate the LEVELS specification (volume on mcap and the dummy, no logs) and run RESET on both the levels model and your log-log model from Exercise 3.

**Written question:** why would comparing the two R² values NOT be a valid way to choose between these models?

In [ ]:
m_lvl = sm.OLS(df['volume'], sm.add_constant(df[['mcap', 'D_SMI']])).fit()
r_lvl = linear_reset(m_lvl, power=3, use_f=True)
r_log = linear_reset(m2, power=3, use_f=True)
print(f'RESET levels:  F = {r_lvl.fvalue:.1f}, p = {r_lvl.pvalue:.4f}')
print(f'RESET log-log: F = {r_log.fvalue:.2f}, p = {r_log.pvalue:.3f}')

**Answer:** the two models have DIFFERENT dependent variables (volume vs ln volume), so their R² values measure explained variation of different things and are not comparable. RESET, by contrast, asks each model on its own terms whether it misses curvature: the levels form typically fails, the log-log form passes. Theory first, RESET as referee, never R² across different y.

---
# Exercise 6: A lin-log Variant

Regress volume IN MILLIONS (levels) on ln(mcap): a lin-log model. Interpret the slope in one sentence, being precise about units.

**Written question:** when might a lin-log form be economically more natural than log-log?

In [ ]:
y_m = df['volume'] / 1e6
m_ll = sm.OLS(y_m, sm.add_constant(df['ln_mcap'])).fit(cov_type='HC1')
b = m_ll.params['ln_mcap']
print(f'slope = {b:.1f}  →  a 1% larger market cap goes with {b/100:.2f} million MORE daily volume')

**Answer:** in lin-log, a 1% change in x moves y by β/100 UNITS. The form is natural when the outcome is genuinely additive in units (costs in francs, headcount, capacity) while the driver varies in percent. Here log-log remains the better choice, since volume itself is right-skewed.

---
# Exercise 7: The Quadratic Fund Model

Simulate the fund sample with the SAME seed as the lecture (so your numbers match), then estimate alpha on size and size squared with HC1 standard errors. Report both coefficients with t-statistics and state the shape.

```python
rng = np.random.default_rng(42)
funds = pd.DataFrame({'size': rng.uniform(0.05, 2.2, 120)})
funds['alpha'] = -0.2 + 2.4*funds['size'] - 1.5*funds['size']**2 + rng.normal(0, 0.55, 120)
```

In [ ]:
rng = np.random.default_rng(42)
funds = pd.DataFrame({'size': rng.uniform(0.05, 2.2, 120)})
funds['alpha'] = -0.2 + 2.4*funds['size'] - 1.5*funds['size']**2 + rng.normal(0, 0.55, 120)
funds['size2'] = funds['size']**2

mq = sm.OLS(funds['alpha'], sm.add_constant(funds[['size', 'size2']])).fit(cov_type='HC1')
print(mq.summary().tables[1])
print('\npositive size, negative size²: an inverted U (rising, peaking, declining)')

**Note:** the model is nonlinear in x but LINEAR in the parameters, so ordinary OLS estimates it and all t- and F-machinery applies unchanged.

---
# Exercise 8: Turning Point and Marginal Effects

(a) Compute the turning point x* = −β₁/(2β₂) and check whether it lies inside the data range.
(b) Compute the marginal effect β₁ + 2β₂x at sizes 0.2, at x*, and at 1.5 bn.
(c) Write ONE sentence a fund allocator could quote.

In [ ]:
b1, b2 = mq.params['size'], mq.params['size2']
xstar = -b1/(2*b2)
print(f'x* = {xstar:.2f} bn, inside [{funds["size"].min():.2f}, {funds["size"].max():.2f}]: '
      f'{funds["size"].min() < xstar < funds["size"].max()}')
for x0 in (0.2, xstar, 1.5):
    print(f'marginal effect at {x0:.2f} bn: {b1 + 2*b2*x0:+.2f} alpha points per additional bn')

**Allocator sentence (model answer):** performance improves with scale for small funds, peaks at roughly 0.7 bn CHF in our sample, and deteriorates beyond it, so size is an advantage only up to the capacity limit.

---
# Exercise 9: What the Linear Model Would Have Told You

Fit the WRONG model, alpha on size only (no square), and run RESET on it.

**Written question:** what conclusion about fund size would the linear model suggest, why is it misleading, and how does RESET warn you?

In [ ]:
m_lin = sm.OLS(funds['alpha'], sm.add_constant(funds['size'])).fit(cov_type='HC1')
r_lin = linear_reset(m_lin, power=3, use_f=True)
print(f'linear slope: {m_lin.params["size"]:+.3f} (t = {m_lin.tvalues["size"]:.2f})')
print(f'RESET: F = {r_lin.fvalue:.1f}, p = {r_lin.pvalue:.4f}')

**Answer:** the linear model does not produce a flat slope at all — it produces a strongly significant negative one, $\hat{\beta}_{size} = -0.917$ with $t = -8.24$. Read on its own it says: bigger funds systematically underperform, so allocate to the small ones. That is the misleading part. The single slope is a weighted average of the two branches of the inverted U — the marginal effect is about $+1.0$ per bn at a size of 0.3 bn and about $-3.0$ per bn at 2.0 bn — and averaging them hides both the positive branch and the capacity peak at roughly 0.7 bn. A significant coefficient from a misspecified model is a confident answer to the wrong question. RESET rejects the linear form decisively ($F \approx 26$, $p < 0.001$) because the omitted curvature shows up in the powers of the fitted values. This is the V9 lesson meeting the V10 tool: RESET detects, the quadratic repairs.

---
# Exercise 10: The Reporting Memo (Solution Sketch)

A model answer — fill in your own figures, because the cross-section is downloaded live and market caps and turnover move. Trading volume scales roughly one for one with company size: a 1% larger market cap goes with about [`m1.params['ln_mcap']`]% more daily volume. Holding size constant, SMI membership adds about [`100*(np.exp(m1.params['D_SMI'])-1)`]% additional volume, consistent with index-tracking flows. For funds, performance follows an inverted U in size: growing helps small funds, the advantage disappears around 0.7 bn CHF, and scale hurts beyond it, at about minus two alpha points per additional billion for a 1.5 bn fund. We therefore report marginal effects at representative sizes rather than a single size effect. The log-log form was chosen because volume and market cap are multiplicative, heavily skewed quantities, and the quadratic because theory and the RESET test both indicated curvature. All estimates use robust standard errors; conventional significance holds throughout.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*